## 01 Data Preparation

This notebook prepares the raw course dataset for curriculum-overlap detection.

The notebook has two goals:

1. **Thesis goal:** create a transparent, defensible course representation for embedding-based overlap detection.
2. **PoC goal:** produce a clean and reusable dataset that later notebooks and the final UI can use without ad hoc fixes.

This notebook is treated as the **source of truth** for downstream work.

Main tasks:
- load and validate the raw JSON dataset
- audit dataset structure and quality
- flatten nested fields into explicit columns
- normalize text conservatively
- create embedding-ready text variants
- prepare fair review text for manual labeling
- generate diagnostics that justify preprocessing choices
- save the cleaned dataset for later notebooks

Main output:
- `data/courses_clean.csv`

### Final representation strategy

This notebook creates three embedding variants:

- **`embedding_text_main`**  
  a conservative core representation of course meaning

- **`embedding_text_enriched`**  
  a broader representation that preserves additional curriculum signals when they exist

- **`embedding_text_full`**  
  the most complete representation, extending the enriched variant with assessment criteria text

This design reflects a deliberate methodological choice:

- the **main** variant should remain stable and comparable across the full dataset
- the **enriched** variant should preserve sparse but potentially meaningful signals such as learning outcomes, teamwork-related competences, multidisciplinary competences, investigations/information retrieval, and prerequisite relations
- the **full** variant should test whether course-specific assessment criteria provide additional semantic signal for overlap detection

The purpose is not to assume that broader text is always better.  
The purpose is to test whether these additional curriculum signals improve overlap detection while keeping the benchmark representation clean, interpretable, and auditable.

### Dataset note

This dataset includes course descriptions from more than one institution.  
That choice was made to improve overlap diversity in the benchmark corpus, because a single-institution set produced too few meaningful overlap cases for robust comparison and threshold evaluation.

This notebook does **not** treat institution as part of course meaning in the embedding text.  
Instead, institution is preserved as reference metadata and included in human-readable review text where it helps inspection.

In [40]:
# Cell 01 - Import libraries and define project paths

import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"

RAW_JSON_PATH = DATA_DIR / "jamk_courses.json"
CLEAN_CSV_PATH = DATA_DIR / "courses_clean.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())
print("Raw JSON path:", RAW_JSON_PATH.resolve())
print("Clean CSV path:", CLEAN_CSV_PATH.resolve())
print("Raw JSON exists:", RAW_JSON_PATH.exists())

Project root: C:\Users\user\Downloads\curriculum-overlap-poc
Data directory: C:\Users\user\Downloads\curriculum-overlap-poc\data
Raw JSON path: C:\Users\user\Downloads\curriculum-overlap-poc\data\jamk_courses.json
Clean CSV path: C:\Users\user\Downloads\curriculum-overlap-poc\data\courses_clean.csv
Raw JSON exists: True


### 1. Load and validate the raw dataset

This section loads the raw course JSON and checks that the file has the expected top-level structure:
- a list of courses
- each course represented as a dictionary

The purpose is not only to load data, but also to confirm that the raw input is structurally suitable for later processing.

In [41]:
# Cell 02 - Load raw JSON safely

if not RAW_JSON_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_JSON_PATH.resolve()}")

with open(RAW_JSON_PATH, "r", encoding="utf-8") as f:
    courses = json.load(f)

print("Dataset type:", type(courses))
print("Number of courses:", len(courses))

if not isinstance(courses, list):
    raise TypeError("Expected the raw dataset to be a list of course dictionaries.")

if len(courses) == 0:
    raise ValueError("The raw dataset is empty.")

print("First item type:", type(courses[0]))

if not all(isinstance(item, dict) for item in courses):
    raise TypeError("All items in the raw dataset should be dictionaries.")

Dataset type: <class 'list'>
Number of courses: 50
First item type: <class 'dict'>


In [42]:
# Cell 03 - Preview one raw course example

courses[0]

{'institution': 'JAMK',
 'degree_programme': 'Information and Communication Technology',
 'course_name': 'Data Networks',
 'course_unit_code': 'TT00CD70',
 'credits': 5,
 'teaching_language': ['Finnish', 'English'],
 'responsible_person': ['Pasi Hyytiäinen'],
 'objective': ['You learn the structure and protocols of data networks used by computers and other end devices.',
  'You know the principles of planning a data network.',
  'You are familiar with protocols and data transmission principles used on the Internet.',
  'You know the layers of the OSI model and identify the technologies and protocols used on each layer.',
  'You can design a local area network using switches, routers, workstations, and servers.',
  'You are able to design cabling, switching topology, and routing scheme to run the data network.',
  'You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations.',
  'You can change, handle, and save the configuration us

In [43]:
# Cell 04 - Convert raw JSON list into a dataframe

raw_df = pd.DataFrame(courses)

print("Raw dataframe shape:", raw_df.shape)
raw_df.head(3)

Raw dataframe shape: (50, 13)


,institution,degree_programme,course_name,course_unit_code,credits,teaching_language,responsible_person,objective,competences,learning_outcomes,content,prerequisites,assessment
0,JAMK,Information and Communication Technology,Data Networks,TT00CD70,5,"[Finnish, English]",[Pasi Hyytiäinen],"[You learn the structure and protocols of data networks used by computers and other end devices., You know the principles of planning a data network., You a...","{'knowledge_and_understanding': ['You know the main modeling methods of data networks.', 'You have knowledge and understanding of data networks and their ef...",[],"[In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication., You will mast...",[Linux Basics],"{'grading_scale': '1-5', 'criteria': {'1': 'The student knows the theory on network protocols sufficiently and is able to recognize some sections of existin..."
1,JAMK,Information and Communication Technology,Linux Basics,TT00CD71,4,"[Finnish, English]","[Juho Pekki, Teemu Siikaniemi]","[After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage ...",{'knowledge_and_understanding': ['You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing ...,[],"[In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be a...",[],"{'grading_scale': '1-5', 'criteria': {'1': 'You are familiar with the basic concepts of the Linux operating system and can use basic commands. You know how ..."
2,JAMK,Information and Communication Technology,Servers and containers,TT00CD72,5,"[Finnish, English]","[Juho Pekki, Teemu Siikaniemi]",[You understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud computing. You are able...,{'knowledge_and_understanding': ['You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service m...,[],"[In this course, you will learn the basic concepts of servers and cloud computing, and differentiate between different cloud service models. You will master...","[Linux Basics, Data Networks, Databases]","{'grading_scale': '1-5', 'criteria': {'1': 'You are to some extent familiar with the basic concepts of the servers and virtualization and know how to create..."


In [44]:
# Cell 05 - Show raw columns

print("Raw columns:")
for col in raw_df.columns:
    print("-", col)

Raw columns:
- institution
- degree_programme
- course_name
- course_unit_code
- credits
- teaching_language
- responsible_person
- objective
- competences
- learning_outcomes
- content
- prerequisites
- assessment


### 2. Initial structural audit

Before building any embedding text, it is important to inspect the raw dataset for:
- missing fields
- empty-but-not-null values
- duplicate identifiers
- unexpected datatypes

This matters for both thesis transparency and PoC reliability.

In [45]:
# Cell 06 - Check true missing values recognized by pandas

true_missing_summary = raw_df.isna().sum().sort_values(ascending=False)
true_missing_summary

institution           0
degree_programme      0
course_name           0
course_unit_code      0
credits               0
teaching_language     0
responsible_person    0
objective             0
competences           0
learning_outcomes     0
content               0
prerequisites         0
assessment            0
dtype: int64

In [46]:
# Cell 07 - Define helper functions for structural checks

def is_empty_list(value):
    return isinstance(value, list) and len(value) == 0

def is_empty_string(value):
    return isinstance(value, str) and value.strip() == ""

def safe_get_dict_value(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default

def get_list_from_dict(d, key):
    value = safe_get_dict_value(d, key, [])
    return value if isinstance(value, list) else []

def get_string_from_dict(d, key):
    value = safe_get_dict_value(d, key, "")
    return value if isinstance(value, str) else ""

def get_nested_string_from_dict(d, parent_key, child_key):
    parent = safe_get_dict_value(d, parent_key, {})
    if isinstance(parent, dict):
        value = parent.get(child_key, "")
        return value if isinstance(value, str) else ""
    return ""

In [47]:
# Cell 08 - Check empty-but-not-null values inside nested structures

empty_checks = {
    "empty_institution": raw_df["institution"].apply(is_empty_string).sum(),
    "empty_prerequisites": raw_df["prerequisites"].apply(is_empty_list).sum(),
    "empty_learning_outcomes": raw_df["learning_outcomes"].apply(is_empty_list).sum(),
    "empty_objective": raw_df["objective"].apply(is_empty_list).sum(),
    "empty_content": raw_df["content"].apply(is_empty_list).sum(),
    "empty_responsible_person": raw_df["responsible_person"].apply(is_empty_list).sum(),
    "empty_teaching_language": raw_df["teaching_language"].apply(is_empty_list).sum(),
    "empty_competences_knowledge_and_understanding": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "knowledge_and_understanding"))
    ).sum(),
    "empty_competences_engineering_practice": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "engineering_practice"))
    ).sum(),
    "empty_competences_applying_technology_to_practice": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "applying_technology_to_practice"))
    ).sum(),
    "empty_competences_multidisciplinary_competence": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "multidisciplinary_competence"))
    ).sum(),
    "empty_competences_communication_and_teamwork": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "communication_and_teamwork"))
    ).sum(),
    "empty_competences_investigations_and_information_retrieval": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "investigations_and_information_retrieval"))
    ).sum(),
    "empty_assessment_grading_scale": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_string_from_dict(x, "grading_scale"))
    ).sum(),
    "empty_assessment_pass_fail_criteria": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_string_from_dict(x, "pass_fail_criteria"))
    ).sum(),
    "empty_assessment_criteria_1": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_nested_string_from_dict(x, "criteria", "1"))
    ).sum(),
    "empty_assessment_criteria_2": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_nested_string_from_dict(x, "criteria", "2"))
    ).sum(),
    "empty_assessment_criteria_3": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_nested_string_from_dict(x, "criteria", "3"))
    ).sum(),
    "empty_assessment_criteria_4": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_nested_string_from_dict(x, "criteria", "4"))
    ).sum(),
    "empty_assessment_criteria_5": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_nested_string_from_dict(x, "criteria", "5"))
    ).sum(),
}

pd.Series(empty_checks).sort_values(ascending=False)

empty_learning_outcomes                                       49
empty_competences_applying_technology_to_practice             49
empty_competences_multidisciplinary_competence                48
empty_competences_investigations_and_information_retrieval    47
empty_assessment_pass_fail_criteria                           46
empty_competences_communication_and_teamwork                  45
empty_assessment_grading_scale                                12
empty_competences_engineering_practice                        11
empty_assessment_criteria_2                                   10
empty_assessment_criteria_3                                   10
empty_prerequisites                                           10
empty_competences_knowledge_and_understanding                 10
empty_assessment_criteria_5                                   10
empty_assessment_criteria_1                                   10
empty_assessment_criteria_4                                   10
empty_responsible_person 

In [48]:
# Cell 09 - Check duplicate course codes

duplicate_codes = raw_df[
    raw_df["course_unit_code"].duplicated(keep=False)
].sort_values("course_unit_code")

duplicate_codes[["course_unit_code", "course_name", "institution"]]

,course_unit_code,course_name,institution


In [49]:
# Cell 10 - Check duplicate course names

duplicate_names = raw_df[
    raw_df["course_name"].duplicated(keep=False)
].sort_values(["course_name", "institution", "course_unit_code"])

duplicate_names[["course_unit_code", "course_name", "institution"]]

,course_unit_code,course_name,institution
10,TT00CD83,Data Structures and Algorithms,JAMK
49,TT00DB42,Data Structures and Algorithms,Turku AMK
9,TT00CD81,Databases,JAMK
44,TE00CS89,Databases,Turku AMK
20,TT00CD94,Mobile Application Development,JAMK
48,TT00DS75,Mobile Application Development,Turku AMK


In [50]:
# Cell 11 - Check datatype consistency of important raw fields

datatype_audit = pd.DataFrame({
    "column": raw_df.columns,
    "python_types_seen": [
        sorted({type(v).__name__ for v in raw_df[col]})
        for col in raw_df.columns
    ]
})

datatype_audit

,column,python_types_seen
0,institution,[str]
1,degree_programme,[str]
2,course_name,[str]
3,course_unit_code,[str]
4,credits,[int]
5,teaching_language,[list]
6,responsible_person,[list]
7,objective,[list]
8,competences,[dict]
9,learning_outcomes,[list]


### Interpretation of the initial structural audit

The dataset remains structurally suitable for downstream preprocessing.

The current version includes an `institution` field in addition to the original course structure.  
This is useful for inspection and review, especially because identical or near-identical course names may now occur across institutions.

No duplicate course codes should appear in the cleaned benchmark identifiers.  
However, duplicate course names across institutions are expected and do not automatically indicate semantic equivalence.

The audit also shows that several curriculum fields remain sparse, especially:
- learning outcomes
- applying technology to practice
- investigations and information retrieval
- multidisciplinary competence
- communication and teamwork

This is acceptable for the current thesis design.  
The notebook does not try to artificially fill such fields. Instead, it preserves what exists, documents coverage explicitly, and tests whether sparse but meaningful curriculum signals improve overlap detection in later notebooks.

### 3. Flatten nested fields into explicit columns

The raw JSON contains nested lists and dictionaries.  
This section converts those structures into explicit columns that are easier to inspect, clean, reuse, and save.

This also makes the preprocessing steps auditable: later notebooks can trace exactly which textual signals were included in the final representation.

In [51]:
# Cell 12 - Copy raw dataframe and flatten nested fields

df = raw_df.copy()

df["competences_knowledge_and_understanding"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "knowledge_and_understanding")
)

df["competences_engineering_practice"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "engineering_practice")
)

df["competences_applying_technology_to_practice"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "applying_technology_to_practice")
)

df["competences_multidisciplinary_competence"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "multidisciplinary_competence")
)

df["competences_communication_and_teamwork"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "communication_and_teamwork")
)

df["competences_investigations_and_information_retrieval"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "investigations_and_information_retrieval")
)

df["assessment_grading_scale"] = df["assessment"].apply(
    lambda x: get_string_from_dict(x, "grading_scale")
)

df["assessment_pass_fail_criteria"] = df["assessment"].apply(
    lambda x: get_string_from_dict(x, "pass_fail_criteria")
)

for level in ["1", "2", "3", "4", "5"]:
    df[f"assessment_criteria_{level}"] = df["assessment"].apply(
        lambda x, level=level: get_nested_string_from_dict(x, "criteria", level)
    )

df[[
    "institution",
    "course_name",
    "competences_knowledge_and_understanding",
    "competences_engineering_practice",
    "competences_applying_technology_to_practice",
    "competences_multidisciplinary_competence",
    "competences_communication_and_teamwork",
    "competences_investigations_and_information_retrieval",
    "learning_outcomes",
    "assessment_grading_scale",
    "assessment_pass_fail_criteria",
    "assessment_criteria_1",
    "assessment_criteria_2",
    "assessment_criteria_3",
    "assessment_criteria_4",
    "assessment_criteria_5",
]].head(3)

,institution,course_name,competences_knowledge_and_understanding,competences_engineering_practice,competences_applying_technology_to_practice,competences_multidisciplinary_competence,competences_communication_and_teamwork,competences_investigations_and_information_retrieval,learning_outcomes,assessment_grading_scale,assessment_pass_fail_criteria,assessment_criteria_1,assessment_criteria_2,assessment_criteria_3,assessment_criteria_4,assessment_criteria_5
0,JAMK,Data Networks,"[You know the main modeling methods of data networks., You have knowledge and understanding of data networks and their effects., You know the basic principl...",[You can apply data network skills to communication between devices.],[],[],[],[],[],1-5,The student has sufficient knowledge of the theory on network protocols and is able to recognize some sections of existing solutions. The student can design...,The student knows the theory on network protocols sufficiently and is able to recognize some sections of existing solutions. The student can design and impl...,The student knows the theory on network protocols satisfactorily and is able to recognize existing solutions. The student can design and implement some part...,The student has good knowledge of the theory on network protocols and is able to analyze existing solutions. The student is able to design and implement mos...,The student masters the theory on network protocols very well and is able to analyze existing solutions. The student is able to plan and implement almost al...,The student has excellent command of the theory on network protocols and is able to analyze existing solutions and create new solutions. The student is able...
1,JAMK,Linux Basics,[You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.],[You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions...,[],[],[],[],[],1-5,,"You are familiar with the basic concepts of the Linux operating system and can use basic commands. You know how to create user groups, add users to them, an...",You understand the meaning of permissions and you are able to apply permissions in the management of filesystems. You know how to use simple command-line to...,You know how to use groups to restrict access and manage user permissions. You understand the basics of package management and know how to install and updat...,You know how to use groups to restrict access and manage user permissions effectively. You know how to use various commands proficiently. You understand the...,"You know how to comprehensively apply user groups and permission management, and combine various command-line tools to increase productivity. You know how t..."
2,JAMK,Servers and containers,[You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service models. You understand the core pr...,"[You are able to utilize basic cloud services and create virtual machines. You are able to build, run, and manage containers, and use orchestration tools to...",[],[],[],[],[],1-5,,You are to some extent familiar with the basic concepts of the servers and virtualization and know how to create your own virtual machines. You know how to ...,You have satisfactory understanding of the basic concepts of the servers and virtualization and know how to create your own virtual machines. You know how t...,You have a good understanding of the basic concepts of the servers and virtualization and know how to create your own virtual machines. You know how to effe...,You have a proficient understanding of the basic concepts of the servers and virtualization and know how to create your own virtual machines. You know how t...,You master the basic concepts of the servers and virtualization and know how to create your own virtual machines. You master how to manage stack

### 4. Conservative text cleaning helpers

This notebook uses **conservative preprocessing**.

Included:
- whitespace normalization
- simple bullet cleanup
- safe joining of list fields into readable text
- light boundary preservation between list items when needed

Avoided on purpose:
- stop-word removal
- stemming or lemmatization
- aggressive phrase stripping
- lowercasing of final embedding text
- injection of full prerequisite course descriptions

The goal is to preserve course meaning as faithfully as possible while avoiding unnecessary preprocessing choices that would be difficult to justify in a thesis setting.

In [52]:
# Cell 13 - Define text cleaning helper functions

def normalize_whitespace(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def remove_leading_bullets(text: str) -> str:
    text = re.sub(r"^[\-\*\u2022]+\s*", "", text)
    return text.strip()

def clean_text_value(value) -> str:
    """
    Clean a single value into a compact readable string.
    Keeps natural language intact for embedding and standard review use.
    Internal line breaks are collapsed into spaces.
    """
    if value is None:
        return ""

    if not isinstance(value, str):
        value = str(value)

    text = value.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = remove_leading_bullets(text)
    text = normalize_whitespace(text)

    return text

def clean_multiline_text_value(value) -> str:
    """
    Clean a value while preserving intentional internal line breaks.
    Useful for already-structured text blocks such as assessment criteria.
    """
    if value is None:
        return ""

    if not isinstance(value, str):
        value = str(value)

    text = value.replace("\xa0", " ")
    text = text.replace("\t", " ")

    lines = text.split("\n")
    cleaned_lines = []
    for line in lines:
        line = remove_leading_bullets(line)
        line = re.sub(r"[ \r\f\v]+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()

def ensure_sentence_like_boundary(text: str) -> str:
    """
    Add a soft boundary between list items when the item does not already end
    with common sentence punctuation. This preserves readability without
    aggressively rewriting the source text.
    """
    if not text:
        return text

    if re.search(r"[.!?:;]$", text):
        return text

    return text + "."

def join_list_field(value, separator=" "):
    """
    Join list items into one clean string.
    Empty items are removed.
    For sentence-like separators, add a soft boundary so list items do not
    collapse into each other.
    """
    if isinstance(value, list):
        cleaned_items = [clean_text_value(item) for item in value]
        cleaned_items = [item for item in cleaned_items if item]

        if separator == " ":
            cleaned_items = [ensure_sentence_like_boundary(item) for item in cleaned_items]

        return separator.join(cleaned_items).strip()

    if isinstance(value, str):
        return clean_text_value(value)

    return ""

def build_labeled_text(sections):
    """
    Build labeled human-readable text.
    Preserves multiline structure when a value already contains line breaks.
    """
    parts = []
    for label, value in sections:
        if isinstance(value, str) and "\n" in value:
            cleaned_value = clean_multiline_text_value(value)
            if cleaned_value:
                parts.append(f"{label}:\n{cleaned_value}")
        else:
            cleaned_value = clean_text_value(value)
            if cleaned_value:
                parts.append(f"{label}: {cleaned_value}")
    return "\n".join(parts)

def normalize_for_matching(text: str) -> str:
    """
    Lowercased helper text for matching / lookup operations.
    Not intended for final embedding text.
    """
    text = clean_text_value(text).lower()
    text = re.sub(r"[^\w\s\-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def char_len(text) -> int:
    return len(clean_text_value(text))

def word_len(text) -> int:
    cleaned = clean_text_value(text)
    return len(cleaned.split()) if cleaned else 0

### 5. Create cleaned flat text columns

These columns serve two purposes:
- they make the course records readable and inspectable
- they provide the building blocks for embedding text and review text

The key design principle is to preserve **semantically rich curriculum content** while keeping the processing pipeline transparent and reproducible.

In [53]:
# Cell 14 - Create cleaned flat text columns

df["institution_text"] = df["institution"].apply(clean_text_value)
df["degree_programme_text"] = df["degree_programme"].apply(clean_text_value)
df["course_name_text"] = df["course_name"].apply(clean_text_value)
df["course_unit_code_text"] = df["course_unit_code"].apply(clean_text_value)

df["teaching_language_text"] = df["teaching_language"].apply(lambda x: join_list_field(x, separator=", "))
df["responsible_person_text"] = df["responsible_person"].apply(lambda x: join_list_field(x, separator=", "))

df["objective_text"] = df["objective"].apply(lambda x: join_list_field(x, separator=" "))

df["competences_knowledge_text"] = df["competences_knowledge_and_understanding"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_practice_text"] = df["competences_engineering_practice"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_applying_technology_text"] = df["competences_applying_technology_to_practice"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_multidisciplinary_text"] = df["competences_multidisciplinary_competence"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_communication_text"] = df["competences_communication_and_teamwork"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_investigations_text"] = df["competences_investigations_and_information_retrieval"].apply(
    lambda x: join_list_field(x, separator=" ")
)

df["learning_outcomes_text"] = df["learning_outcomes"].apply(lambda x: join_list_field(x, separator=" "))
df["content_text"] = df["content"].apply(lambda x: join_list_field(x, separator=" "))
df["prerequisites_text"] = df["prerequisites"].apply(lambda x: join_list_field(x, separator=", "))

df["assessment_pass_fail_criteria_text"] = df["assessment_pass_fail_criteria"].apply(clean_text_value)

for level in ["1", "2", "3", "4", "5"]:
    df[f"assessment_criteria_{level}_text"] = df[f"assessment_criteria_{level}"].apply(clean_text_value)

def build_assessment_text(row):
    sections = [
        ("Grade 1", row["assessment_criteria_1_text"]),
        ("Grade 2", row["assessment_criteria_2_text"]),
        ("Grade 3", row["assessment_criteria_3_text"]),
        ("Grade 4", row["assessment_criteria_4_text"]),
        ("Grade 5", row["assessment_criteria_5_text"]),
        ("Pass/fail", row["assessment_pass_fail_criteria_text"]),
    ]
    return build_labeled_text(sections)

df["assessment_text"] = df.apply(build_assessment_text, axis=1)

df[[
    "institution",
    "course_name",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_text",
]].head(5)

,institution,course_name,objective_text,competences_knowledge_text,competences_practice_text,competences_applying_technology_text,competences_multidisciplinary_text,competences_communication_text,competences_investigations_text,learning_outcomes_text,content_text,prerequisites_text,assessment_text
0,JAMK,Data Networks,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...,You can apply data network skills to communication between devices.,,,,,,"In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication. You will master...",Linux Basics,Grade 1: The student knows the theory on network protocols sufficiently and is able to recognize some sections of existing solutions. The student can design...
1,JAMK,Linux Basics,"After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage t...",You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.,You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions....,,,,,,"In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be ab...",,"Grade 1: You are familiar with the basic concepts of the Linux operating system and can use basic commands. You know how to create user groups, add users to..."
2,JAMK,Servers and containers,You understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud computing. You are able ...,You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service models. You understand the core pri...,"You are able to utilize basic cloud services and create virtual machines. You are able to build, run, and manage containers, and use orchestration tools to ...",,,,,,"In this course, you will learn the basic concepts of servers and cloud computing, and differentiate between different cloud service models. You will master ...","Linux Basics, Data Networks, Databases",Grade 1: You are to some extent familiar with the basic concepts of the servers and virtualization and know how to create your own virtual machines. You kno...
3,JAMK,Windows Basics,You understand and know how to use Windows management tools and resource monitoring tools. You are able to diagnose and solve the most common problems on a ...,You know the key basic features and tools of the Windows operating system.,You know how to manage the most important functions of the Windows system. You know how to manage the file system and user account rights. You know how to i...,,,,,,In this course you will learn how to use Windows management tools and monitoring tools effectively. You will be able to diagnose and solve common problems o...,,Grade 1: You know the basic features of the Windows operating system and know how to use some management and monitoring tools. There are shortcomings in tro...
4,JAMK,Scripting and Automatization,You understand the benefits of automation related to the management of operating system and the principles of the operation of command interpreters. You kno...,You understand the basic methods of automation and know how to use this in basic tasks. You know the basic concepts of automation.,You recognize the importance of automation in IT. You know how to apply automation in different environments.,,,,,,"In this course, you will learn the benefits of automating operatin

In [54]:
# Cell 15 - Create helper normalized columns for matching and lightweight rule-based use

df["course_name_normalized"] = df["course_name"].apply(normalize_for_matching)
df["prerequisites_text_normalized"] = df["prerequisites_text"].apply(normalize_for_matching)

df[[
    "course_name",
    "course_name_normalized",
    "prerequisites_text",
    "prerequisites_text_normalized"
]].head(5)

,course_name,course_name_normalized,prerequisites_text,prerequisites_text_normalized
0,Data Networks,data networks,Linux Basics,linux basics
1,Linux Basics,linux basics,,
2,Servers and containers,servers and containers,"Linux Basics, Data Networks, Databases",linux basics data networks databases
3,Windows Basics,windows basics,,
4,Scripting and Automatization,scripting and automatization,"Linux Basics, Windows Basics",linux basics windows basics


In [55]:
# Cell 16 - Define lightweight semantic cue helpers for enriched and full embedding text

def prefix_if_text_exists(prefix: str, text: str) -> str:
    cleaned = clean_text_value(text)
    if not cleaned:
        return ""
    return f"{prefix}{cleaned}"

def prefix_if_multiline_text_exists(prefix: str, text: str) -> str:
    """
    Preserve internal line breaks when the text is already intentionally structured
    across multiple lines, such as assessment criteria blocks.
    """
    if text is None:
        return ""

    if not isinstance(text, str):
        text = str(text)

    cleaned = text.replace("\xa0", " ")
    cleaned = cleaned.replace("\t", " ")
    cleaned = re.sub(r"[ \r\f\v]+", " ", cleaned)
    cleaned = re.sub(r"\n+", "\n", cleaned).strip()

    if not cleaned:
        return ""

    return f"{prefix}{cleaned}"

def build_prerequisites_phrase(text: str) -> str:
    return prefix_if_text_exists("Prerequisites include: ", text)

def build_learning_outcomes_phrase(text: str) -> str:
    return prefix_if_text_exists("Learning outcomes include: ", text)

def build_applying_technology_phrase(text: str) -> str:
    return prefix_if_text_exists("Applying technology to practice includes: ", text)

def build_multidisciplinary_phrase(text: str) -> str:
    return prefix_if_text_exists("Multidisciplinary competence includes: ", text)

def build_communication_phrase(text: str) -> str:
    return prefix_if_text_exists("Communication and teamwork includes: ", text)

def build_investigations_phrase(text: str) -> str:
    return prefix_if_text_exists("Investigations and information retrieval includes: ", text)

def build_assessment_phrase(text: str) -> str:
    return prefix_if_multiline_text_exists("Assessment criteria include:\n", text)

### 6. Build text variants on purpose

A single text representation is rarely enough for a defensible benchmark.

This notebook creates three embedding variants:

- **`embedding_text_main`**  
  Primary benchmark representation. This is the default semantic course representation used for overlap detection.

- **`embedding_text_enriched`**  
  Extended variant that tests whether sparse competence fields, learning outcomes, and prerequisite information add useful curricular signal.

- **`embedding_text_full`**  
  A more complete variant that extends the enriched representation with assessment criteria text.

#### Embedding text design

The embedding variants are constructed from selected cleaned text fields.

##### `embedding_text_main`
This column contains the core semantic course representation:
- `course_name`
- `objective`
- `competences.knowledge_and_understanding`
- `competences.engineering_practice`
- `content`

##### `embedding_text_enriched`
This column extends the main representation with additional curriculum signals:
- all fields from `embedding_text_main`
- `competences.applying_technology_to_practice`
- `competences.multidisciplinary_competence`
- `competences.communication_and_teamwork`
- `competences.investigations_and_information_retrieval`
- `learning_outcomes`
- `prerequisites`

##### `embedding_text_full`
This column extends the enriched representation with:
- all fields from `embedding_text_enriched`
- `assessment_text`

This split is intentional:

- **`embedding_text_main`** represents what the course is.
- **`embedding_text_enriched`** represents what the course is plus surrounding curriculum signals.
- **`embedding_text_full`** tests whether course-specific assessment criteria provide useful additional signal.

#### Reducing structural bias

The embedding variants are intentionally built **without heavy section labels** such as
`Course name:`, `Objective:`, or `Content:`.

This design reduces repeated structural boilerplate shared by all courses.

However, a few fields receive **light semantic cues** where the relationship matters for interpretation:
- prerequisites
- learning outcomes
- sparse competence categories included only in the enriched variant
- assessment criteria included only in the full variant

By contrast, the human review text remains labeled, because labels improve readability during manual inspection and labeling.

Institution is intentionally excluded from embedding text.  
It is preserved in review text and metadata so that same-name cross-institution courses can be inspected clearly.

In [56]:
# Cell 17 - Define text builders for embedding and review use

def build_unlabeled_text(parts):
    cleaned_parts = []
    for part in parts:
        if part is None:
            continue
        if not isinstance(part, str):
            part = str(part)

        cleaned = part.replace("\xa0", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = re.sub(r"[ \r\f\v]+", " ", cleaned)
        cleaned = re.sub(r"\n+", "\n", cleaned).strip()

        if cleaned:
            cleaned_parts.append(cleaned)

    return "\n".join(cleaned_parts)

def build_embedding_text_main(row):
    return build_unlabeled_text([
        row["course_name_text"],
        row["objective_text"],
        row["competences_knowledge_text"],
        row["competences_practice_text"],
        row["content_text"],
    ])

def build_embedding_text_enriched(row):
    return build_unlabeled_text([
        row["course_name_text"],
        row["objective_text"],
        row["competences_knowledge_text"],
        row["competences_practice_text"],
        row["content_text"],
        build_applying_technology_phrase(row["competences_applying_technology_text"]),
        build_multidisciplinary_phrase(row["competences_multidisciplinary_text"]),
        build_communication_phrase(row["competences_communication_text"]),
        build_investigations_phrase(row["competences_investigations_text"]),
        build_learning_outcomes_phrase(row["learning_outcomes_text"]),
        build_prerequisites_phrase(row["prerequisites_text"]),
    ])

def build_embedding_text_full(row):
    return build_unlabeled_text([
        row["course_name_text"],
        row["objective_text"],
        row["competences_knowledge_text"],
        row["competences_practice_text"],
        row["content_text"],
        build_applying_technology_phrase(row["competences_applying_technology_text"]),
        build_multidisciplinary_phrase(row["competences_multidisciplinary_text"]),
        build_communication_phrase(row["competences_communication_text"]),
        build_investigations_phrase(row["competences_investigations_text"]),
        build_learning_outcomes_phrase(row["learning_outcomes_text"]),
        build_prerequisites_phrase(row["prerequisites_text"]),
        build_assessment_phrase(row["assessment_text"]),
    ])

def build_review_text_compact(row):
    sections = [
        ("Institution", row["institution_text"]),
        ("Course code", row["course_unit_code_text"]),
        ("Course name", row["course_name_text"]),
        ("Objective", row["objective_text"]),
        ("Content", row["content_text"]),
        ("Prerequisites", row["prerequisites_text"]),
    ]
    return build_labeled_text(sections)

def build_review_text_full(row):
    sections = [
        ("Institution", row["institution_text"]),
        ("Course code", row["course_unit_code_text"]),
        ("Course name", row["course_name_text"]),
        ("Objective", row["objective_text"]),
        ("Knowledge and understanding", row["competences_knowledge_text"]),
        ("Engineering practice", row["competences_practice_text"]),
        ("Applying technology to practice", row["competences_applying_technology_text"]),
        ("Multidisciplinary competence", row["competences_multidisciplinary_text"]),
        ("Communication and teamwork", row["competences_communication_text"]),
        ("Investigations and information retrieval", row["competences_investigations_text"]),
        ("Learning outcomes", row["learning_outcomes_text"]),
        ("Content", row["content_text"]),
        ("Prerequisites", row["prerequisites_text"]),
        ("Assessment grading scale", row["assessment_grading_scale"]),
        ("Assessment", row["assessment_text"]),
    ]
    return build_labeled_text(sections)

In [57]:
# Cell 18 - Create embedding and review text variants

df["embedding_text_main"] = df.apply(build_embedding_text_main, axis=1)
df["embedding_text_enriched"] = df.apply(build_embedding_text_enriched, axis=1)
df["embedding_text_full"] = df.apply(build_embedding_text_full, axis=1)

df["review_text_compact"] = df.apply(build_review_text_compact, axis=1)
df["review_text_full"] = df.apply(build_review_text_full, axis=1)

print(df.loc[0, "embedding_text_main"])

Data Networks
You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles of network design and how to 

In [59]:
# Cell 19 - Preview embedding and review variants for one course

preview_columns = [
    "embedding_text_main",
    "embedding_text_enriched",
    "embedding_text_full",
    "review_text_full",
]

for col in preview_columns:
    print("=" * 100)
    print(col)
    print("-" * 100)
    print(df.loc[0, col])
    print()

embedding_text_main
----------------------------------------------------------------------------------------------------
Data Networks
You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
You know the main modeling methods of data networks. You have kn

### 7. Prepare fair review text for manual labeling

Manual labeling should not be based on a weaker representation than the one given to the models.

For that reason, this notebook prepares explicit review text fields that can later be shown in Notebook 02.

These fields are intended for human inspection, not for direct embedding:
- they keep labels for clarity
- they show the course structure more explicitly
- they help reduce annotation mistakes during pair review

Because the dataset contains courses from multiple institutions, review text also includes institution metadata.  
This helps avoid confusion when same-name courses from different institutions are compared manually.

In [60]:
# Cell 20 - Preview one human-readable full review text

print(df.loc[0, "review_text_full"])

Institution: JAMK
Course code: TT00CD70
Course name: Data Networks
Objective: You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
Knowledge and understanding: You know the main modeling methods of data networks. You have knowledge and understanding of

### 8. Field-level audit

A strong preprocessing notebook should not only create text, but also explain **where the text comes from** and **how representation choices may affect later results**.

This section audits:
- coverage of important fields
- character and word contribution by field
- whether enrichment meaningfully expands the representation
- whether assessment meaningfully expands the representation
- whether some fields are repetitive or sparse
- whether the enriched and full representations are activated by only a few special cases or by broader dataset patterns

This section is especially useful later when justifying preprocessing decisions in the thesis.

A key point in this notebook is that some curriculum signals are **structurally sparse but semantically important**.  
For that reason, both coverage and qualitative inspection are needed: low coverage alone is not sufficient reason to discard a field if it meaningfully represents a subset of courses.

In [61]:
# Cell 21 - Coverage audit of cleaned core fields

coverage_rows = []

for col in [
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_grading_scale",
    "assessment_text",
    "embedding_text_main",
    "embedding_text_enriched",
    "embedding_text_full",
    "review_text_full",
]:
    non_empty_count = (df[col].fillna("").str.strip() != "").sum()
    coverage_rows.append({
        "column": col,
        "non_empty_rows": int(non_empty_count),
        "empty_rows": int(len(df) - non_empty_count),
        "coverage_ratio": round(non_empty_count / len(df), 4),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values(
    by=["coverage_ratio", "column"],
    ascending=[False, True]
).reset_index(drop=True)

coverage_df

,column,non_empty_rows,empty_rows,coverage_ratio
0,content_text,50,0,1.00
1,embedding_text_enriched,50,0,1.00
2,embedding_text_full,50,0,1.00
3,embedding_text_main,50,0,1.00
4,objective_text,50,0,1.00
5,review_text_full,50,0,1.00
6,assessment_text,41,9,0.82
7,competences_knowledge_text,40,10,0.80
8,prerequisites_text,40,10,0.80
9,competences_practice_text,39,11,0.78


### Interpretation of field coverage

The coverage audit shows a clear difference between core descriptive content and supplementary curriculum metadata.

The dataset remains strong for overlap detection because the most important semantic fields — `objective_text` and `content_text` — are present for all 50 courses. As a result, all embedding variants remain non-empty across the full corpus.

At the same time, several supporting fields are much sparser:
- competences fields
- learning outcomes
- assessment fields

This is expected in the current mixed-institution dataset. Some courses provide richer structured curriculum metadata than others, but the benchmark remains usable because the main semantic description of each course is still available.

For this reason:
- `embedding_text_main` remains the most stable benchmark representation
- `embedding_text_enriched` remains a controlled extension for additional curriculum signals
- `embedding_text_full` remains an experimental variant whose usefulness must be tested empirically rather than assumed

In [62]:
# Cell 22 - Field length audit by course

length_audit_df = df[[
    "institution",
    "course_unit_code",
    "course_name",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_text",
    "embedding_text_main",
    "embedding_text_enriched",
    "embedding_text_full",
    "review_text_full",
]].copy()

text_cols_for_length = [
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_text",
    "embedding_text_main",
    "embedding_text_enriched",
    "embedding_text_full",
    "review_text_full",
]

for col in text_cols_for_length:
    length_audit_df[f"{col}_char_len"] = length_audit_df[col].apply(char_len)
    length_audit_df[f"{col}_word_len"] = length_audit_df[col].apply(word_len)

length_summary = length_audit_df[
    [c for c in length_audit_df.columns if c.endswith("_char_len") or c.endswith("_word_len")]
].describe()

length_summary

,objective_text_char_len,objective_text_word_len,competences_knowledge_text_char_len,competences_knowledge_text_word_len,competences_practice_text_char_len,competences_practice_text_word_len,competences_applying_technology_text_char_len,competences_applying_technology_text_word_len,competences_multidisciplinary_text_char_len,competences_multidisciplinary_text_word_len,competences_communication_text_char_len,competences_communication_text_word_len,competences_investigations_text_char_len,competences_investigations_text_word_len,learning_outcomes_text_char_len,learning_outcomes_text_word_len,content_text_char_len,content_text_word_len,prerequisites_text_char_len,prerequisites_text_word_len,assessment_text_char_len,assessment_text_word_len,embedding_text_main_char_len,embedding_text_main_word_len,embedding_text_enriched_char_len,embedding_text_enriched_word_len,embedding_text_full_char_len,embedding_text_full_word_len,review_text_full_char_len,review_text_full_word_len
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000,50.00000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,431.280000,64.140000,120.940000,17.880000,127.800000,19.700000,1.900000,0.30000,7.760000,1.200000,16.840000,2.700000,11.020000,1.900000,6.440000,0.980000,797.900000,113.420000,37.54000,4.760000,1169.88000,184.680000,1506.020000,218.08000,1616.660000,232.500000,2811.140000,419.640000,2925.840000,433.140000
std,236.309296,33.403538,91.030026,13.112761,103.372913,16.378495,13.435029,2.12132,41.664628,6.493325,55.570275,8.730079,45.850907,7.872064,45.537677,6.929646,372.839586,48.392987,31.83004,3.600227,702.55274,108.060346,550.003951,75.47478,612.904347,85.708393,1072.285657,164.146868,1080.667261,165.014534
min,185.000000,29.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,122.000000,16.000000,0.00000,0.000000,0.00000,0.000000,368.000000,50.00000,368.000000,50.000000,368.000000,50.000000,446.000000,60.000000
25%,264.750000,40.000000,64.000000,8.250000,69.250000,9.250000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,644.000000,93.250000,21.00000,3.000000,956.25000,161.250000,1209.500000,178.25000,1274.750000,190.750000,2420.250000,367.750000,2555.250000,380.500000
50%,322.000000,48.500000,114.000000,18.500000,99.000000,14.500000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,751.000000,109.500000,28.50000,4.000000,1227.00000,200.000000,1419.000000,209.00000,1507.500000,219.500000,2702.500000,413.000000,2831.500000,428.000000
75%,583.500000,82.750000,177.250000,26.000000,209.000000,32.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,899.000000,129.750000,57.50000,7.000000,1553.75000,236.750000,1727.000000,245.00000,1921.000000,266.750000,3460.750000,503.000000,3588.500000,518.000000
max,1050.000000,147.000000,358.000000,47.000000,403.000000,59.000000,95.000000,15.00000,274.000000,43.000000,284.000000,43.000000,229.000000,39.000000,322.000000,49.000000,2498.000000,311.000000,157.00000,16.000000,2822.00000,416.000000,3382.000000,420.00000,3382.000000,430.000000,4869.000000,731.000000,4989.000000,745.000000


### Note on text length and downstream model context

The text-length diagnostics in this notebook are important for interpreting later benchmark results.

In this corpus, the primary course representations are long enough that short-context embedding models may truncate important parts of the text. This matters especially when comparing models with very different context limits.

As a practical interpretation:
- short-context models may lose part of the course description
- medium-context models are likely to handle most courses adequately
- long-context models can process the full representation without truncation concerns

This issue is discussed and quantified in Notebook 03 when model performance is compared against the final text variants.

The newline structure in the embedding text is also intentional. It preserves soft section boundaries between course name, objectives, competences, content, learning outcomes, prerequisites, and assessment criteria. Downstream notebooks should avoid collapsing these line breaks unless there is a clear methodological reason to do so.

In [63]:
# Cell 23 - Show average contribution by field

field_contribution_rows = []

for col in [
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_text",
]:
    field_contribution_rows.append({
        "field": col,
        "mean_char_len": round(df[col].apply(char_len).mean(), 2),
        "median_char_len": round(df[col].apply(char_len).median(), 2),
        "mean_word_len": round(df[col].apply(word_len).mean(), 2),
        "median_word_len": round(df[col].apply(word_len).median(), 2),
        "non_empty_rows": int((df[col].fillna("").str.strip() != "").sum()),
    })

field_contribution_df = pd.DataFrame(field_contribution_rows).sort_values(
    by="mean_char_len",
    ascending=False
).reset_index(drop=True)

field_contribution_df

,field,mean_char_len,median_char_len,mean_word_len,median_word_len,non_empty_rows
0,assessment_text,1169.88,1227.0,184.68,200.0,41
1,content_text,797.90,751.0,113.42,109.5,50
2,objective_text,431.28,322.0,64.14,48.5,50
3,competences_practice_text,127.80,99.0,19.70,14.5,39
4,competences_knowledge_text,120.94,114.0,17.88,18.5,40
5,prerequisites_text,37.54,28.5,4.76,4.0,40
6,competences_communication_text,16.84,0.0,2.70,0.0,5
7,competences_investigations_text,11.02,0.0,1.90,0.0,3
8,competences_multidisciplinary_text,7.76,0.0,1.20,0.0,2
9,learning_outcomes_text,6.44,0.0,0.98,0.0,1


In [64]:
# Cell 24 - Measure how much the enriched representation expands the main representation

enriched_impact_df = df[[
    "institution",
    "course_unit_code",
    "course_name",
    "learning_outcomes_text",
    "prerequisites_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "embedding_text_main",
    "embedding_text_enriched",
]].copy()

enriched_impact_df["main_char_len"] = enriched_impact_df["embedding_text_main"].apply(char_len)
enriched_impact_df["enriched_char_len"] = enriched_impact_df["embedding_text_enriched"].apply(char_len)
enriched_impact_df["added_chars_from_enrichment"] = (
    enriched_impact_df["enriched_char_len"] - enriched_impact_df["main_char_len"]
)

enriched_impact_df["has_learning_outcomes"] = enriched_impact_df["learning_outcomes_text"].str.strip() != ""
enriched_impact_df["has_prerequisites"] = enriched_impact_df["prerequisites_text"].str.strip() != ""
enriched_impact_df["has_applying_technology"] = enriched_impact_df["competences_applying_technology_text"].str.strip() != ""
enriched_impact_df["has_multidisciplinary"] = enriched_impact_df["competences_multidisciplinary_text"].str.strip() != ""
enriched_impact_df["has_communication"] = enriched_impact_df["competences_communication_text"].str.strip() != ""
enriched_impact_df["has_investigations"] = enriched_impact_df["competences_investigations_text"].str.strip() != ""

enriched_impact_df["num_enriched_signal_sources"] = (
    enriched_impact_df[
        [
            "has_learning_outcomes",
            "has_prerequisites",
            "has_applying_technology",
            "has_multidisciplinary",
            "has_communication",
            "has_investigations",
        ]
    ]
    .sum(axis=1)
)

enriched_impact_df[[
    "institution",
    "course_unit_code",
    "course_name",
    "num_enriched_signal_sources",
    "has_learning_outcomes",
    "has_prerequisites",
    "has_applying_technology",
    "has_multidisciplinary",
    "has_communication",
    "has_investigations",
    "added_chars_from_enrichment"
]].sort_values(
    by=["added_chars_from_enrichment", "num_enriched_signal_sources"],
    ascending=[False, False]
).head(15)

,institution,course_unit_code,course_name,num_enriched_signal_sources,has_learning_outcomes,has_prerequisites,has_applying_technology,has_multidisciplinary,has_communication,has_investigations,added_chars_from_enrichment
31,JAMK,TT00CE09,Auditing and Penetration Testing,3,False,True,False,True,False,True,662
16,JAMK,TT00CD90,Software Design,3,False,True,False,True,True,False,469
22,JAMK,TT00CD96,Graphics Programming,2,True,True,False,False,False,False,456
33,JAMK,TT00CE12,Security Operations Center,2,False,True,False,False,True,False,395
21,JAMK,TT00CD92,Mobile Project,2,False,True,False,False,True,False,362
39,JAMK,TT00CE20,Local Area Networks,2,False,True,False,False,False,True,337
40,JAMK,TT00CE28,Core Networks,2,False,True,False,False,False,True,246
34,JAMK,TT00CD88,Hardening,2,False,True,True,False,False,False,205
30,JAMK,TT00CE06,Machine Learning Project,2,False,True,False,False,True,False,180
29,JAMK,TT00CE05,Data Analytics Project,1,False,False,False,False,True,False,150


In [65]:
# Cell 24B - Summarize which enriched-only signal sources activate across courses

enriched_signal_summary = pd.DataFrame({
    "signal_source": [
        "learning_outcomes",
        "prerequisites",
        "applying_technology_to_practice",
        "multidisciplinary_competence",
        "communication_and_teamwork",
        "investigations_and_information_retrieval",
    ],
    "non_empty_rows": [
        int((df["learning_outcomes_text"].str.strip() != "").sum()),
        int((df["prerequisites_text"].str.strip() != "").sum()),
        int((df["competences_applying_technology_text"].str.strip() != "").sum()),
        int((df["competences_multidisciplinary_text"].str.strip() != "").sum()),
        int((df["competences_communication_text"].str.strip() != "").sum()),
        int((df["competences_investigations_text"].str.strip() != "").sum()),
    ]
})

enriched_signal_summary["coverage_ratio"] = (
    enriched_signal_summary["non_empty_rows"] / len(df)
).round(4)

enriched_signal_summary.sort_values(
    by=["non_empty_rows", "signal_source"],
    ascending=[False, True]
).reset_index(drop=True)

,signal_source,non_empty_rows,coverage_ratio
0,prerequisites,40,0.80
1,communication_and_teamwork,5,0.10
2,investigations_and_information_retrieval,3,0.06
3,multidisciplinary_competence,2,0.04
4,applying_technology_to_practice,1,0.02
5,learning_outcomes,1,0.02


In [66]:
# Cell 24C - Measure how much the full representation expands the enriched representation

full_impact_df = df[[
    "institution",
    "course_unit_code",
    "course_name",
    "assessment_grading_scale",
    "assessment_text",
    "embedding_text_enriched",
    "embedding_text_full",
]].copy()

full_impact_df["enriched_char_len"] = full_impact_df["embedding_text_enriched"].apply(char_len)
full_impact_df["full_char_len"] = full_impact_df["embedding_text_full"].apply(char_len)
full_impact_df["added_chars_from_assessment"] = (
    full_impact_df["full_char_len"] - full_impact_df["enriched_char_len"]
)

full_impact_df["has_assessment_text"] = full_impact_df["assessment_text"].str.strip() != ""

full_impact_df[[
    "institution",
    "course_unit_code",
    "course_name",
    "assessment_grading_scale",
    "has_assessment_text",
    "added_chars_from_assessment"
]].sort_values(
    by=["added_chars_from_assessment", "course_unit_code"],
    ascending=[False, True]
).head(15)

,institution,course_unit_code,course_name,assessment_grading_scale,has_assessment_text,added_chars_from_assessment
21,JAMK,TT00CD92,Mobile Project,1-5,True,2852
6,JAMK,TT00CD78,Frontend Programming,1-5,True,2479
31,JAMK,TT00CE09,Auditing and Penetration Testing,1-5,True,2218
33,JAMK,TT00CE12,Security Operations Center,1-5,True,2148
12,JAMK,TT00CD85,Backend Programming,1-5,True,2106
15,JAMK,TT00CE17,Web Application Security Testing,1-5,True,1989
20,JAMK,TT00CD94,Mobile Application Development,1-5,True,1981
39,JAMK,TT00CE20,Local Area Networks,,True,1933
40,JAMK,TT00CE28,Core Networks,,True,1897
19,JAMK,TT00CD93,Android Application Development,1-5,True,1884


In [67]:
# Cell 24D - Summarize assessment signal coverage across courses

assessment_signal_summary = pd.DataFrame({
    "signal_source": [
        "assessment_text",
        "assessment_pass_fail_criteria_helper",
        "assessment_criteria_1_helper",
        "assessment_criteria_2_helper",
        "assessment_criteria_3_helper",
        "assessment_criteria_4_helper",
        "assessment_criteria_5_helper",
    ],
    "non_empty_rows": [
        int((df["assessment_text"].str.strip() != "").sum()),
        int((df["assessment_pass_fail_criteria_text"].str.strip() != "").sum()),
        int((df["assessment_criteria_1_text"].str.strip() != "").sum()),
        int((df["assessment_criteria_2_text"].str.strip() != "").sum()),
        int((df["assessment_criteria_3_text"].str.strip() != "").sum()),
        int((df["assessment_criteria_4_text"].str.strip() != "").sum()),
        int((df["assessment_criteria_5_text"].str.strip() != "").sum()),
    ]
})

assessment_signal_summary["coverage_ratio"] = (
    assessment_signal_summary["non_empty_rows"] / len(df)
).round(4)

assessment_signal_summary.sort_values(
    by=["non_empty_rows", "signal_source"],
    ascending=[False, True]
).reset_index(drop=True)

,signal_source,non_empty_rows,coverage_ratio
0,assessment_text,41,0.82
1,assessment_criteria_1_helper,40,0.80
2,assessment_criteria_2_helper,40,0.80
3,assessment_criteria_3_helper,40,0.80
4,assessment_criteria_4_helper,40,0.80
5,assessment_criteria_5_helper,40,0.80
6,assessment_pass_fail_criteria_helper,4,0.08


In [68]:
# Cell 25 - Check the courses with the longest representations

length_audit_df[[
    "institution",
    "course_unit_code",
    "course_name",
    "embedding_text_main_char_len",
    "embedding_text_enriched_char_len",
    "embedding_text_full_char_len",
]].sort_values(
    by="embedding_text_full_char_len",
    ascending=False
).head(10)

,institution,course_unit_code,course_name,embedding_text_main_char_len,embedding_text_enriched_char_len,embedding_text_full_char_len
33,JAMK,TT00CE12,Security Operations Center,2326,2721,4869
21,JAMK,TT00CD92,Mobile Project,1587,1949,4801
40,JAMK,TT00CE28,Core Networks,2441,2687,4584
31,JAMK,TT00CE09,Auditing and Penetration Testing,1676,2338,4556
39,JAMK,TT00CE20,Local Area Networks,2257,2594,4527
16,JAMK,TT00CD90,Software Design,2337,2806,4526
6,JAMK,TT00CD78,Frontend Programming,1422,1509,3988
12,JAMK,TT00CD85,Backend Programming,1750,1837,3943
22,JAMK,TT00CD96,Graphics Programming,1943,2399,3764
0,JAMK,TT00CD70,Data Networks,2108,2144,3727


### 9. Repetitiveness audit for administrative boilerplate and rubric text

One risk in curriculum-overlap work is adding text that is highly repetitive across many courses, because that can inflate similarity for the wrong reason.

Assessment-related text is included only in the full variant as a controlled experimental condition. This section performs a lightweight audit of repeated phrases in selected fields.

In [69]:
# Cell 26 - Lightweight repeated-text audit

def get_repeated_exact_values(series, min_count=2):
    cleaned_values = [clean_text_value(v) for v in series if clean_text_value(v)]
    counts = Counter(cleaned_values)
    repeated = [(text, count) for text, count in counts.items() if count >= min_count]
    repeated = sorted(repeated, key=lambda x: (-x[1], x[0]))
    return pd.DataFrame(repeated, columns=["text", "count"])

print("Repeated exact values in assessment_pass_fail_criteria_text:")
repeated_pass_fail_df = get_repeated_exact_values(df["assessment_pass_fail_criteria_text"], min_count=2)
repeated_pass_fail_df.head(10)

Repeated exact values in assessment_pass_fail_criteria_text:


,text,count


In [70]:
# Cell 27 - Lightweight repeated-fragment audit for selected descriptive fields

def split_into_candidate_lines(text):
    cleaned = clean_text_value(text)
    if not cleaned:
        return []
    parts = re.split(r"(?<=[\.\;\:])\s+|\n+", cleaned)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

def repeated_fragments_from_series(series, min_count=3, min_chars=30):
    fragments = []
    for value in series:
        fragments.extend([
            frag for frag in split_into_candidate_lines(value)
            if len(frag) >= min_chars
        ])
    counts = Counter(fragments)
    repeated = [(frag, count) for frag, count in counts.items() if count >= min_count]
    repeated = sorted(repeated, key=lambda x: (-x[1], x[0]))
    return pd.DataFrame(repeated, columns=["fragment", "count"])

print("Repeated fragments in objective_text:")
repeated_objective_fragments_df = repeated_fragments_from_series(df["objective_text"], min_count=3, min_chars=40)
display(repeated_objective_fragments_df.head(15))

print("Repeated fragments in assessment_text:")
repeated_assessment_fragments_df = repeated_fragments_from_series(df["assessment_text"], min_count=3, min_chars=40)
display(repeated_assessment_fragments_df.head(15))

Repeated fragments in objective_text:


,fragment,count
0,"After completing the course, the student can:",3


Repeated fragments in assessment_text:


,fragment,count
0,The student can solve problems with backbone networks.,3
1,The student knows how to plan and implement parts of a small backbone network based on the specification of requirements.,3
2,The student knows how to solve local area network problems.,3
3,The student knows how to use communication and reporting tools in accordance with the instructions.,3
4,You can apply your knowledge to the solutions of practical tasks.,3
5,You have completed all the assignments of the course.,3
6,You have completed almost all the assignments of the course.,3
7,You have completed most of the course assignments.,3
8,You have only completed a part of the course assignments.,3
9,You have only completed a part of the course exercises.,3


In [71]:
# Cell 27B - Show which courses contain repeated assessment fragments

def find_fragment_examples(fragment_df, source_series, id_df, top_n_fragments=10, max_examples_per_fragment=5):
    rows = []

    top_fragments = fragment_df.head(top_n_fragments)["fragment"].tolist()

    for frag in top_fragments:
        mask = source_series.fillna("").apply(lambda x: frag in clean_text_value(x))
        matches = id_df[mask].head(max_examples_per_fragment)

        for _, row in matches.iterrows():
            rows.append({
                "fragment": frag,
                "institution": row["institution"],
                "course_unit_code": row["course_unit_code"],
                "course_name": row["course_name"],
            })

    return pd.DataFrame(rows)

fragment_examples_df = find_fragment_examples(
    fragment_df=repeated_assessment_fragments_df,
    source_series=df["assessment_text"],
    id_df=df[["institution", "course_unit_code", "course_name"]],
    top_n_fragments=10,
    max_examples_per_fragment=5
)

fragment_examples_df.head(30)

,fragment,institution,course_unit_code,course_name
0,The student can solve problems with backbone networks.,JAMK,TT00CE28,Core Networks
1,The student knows how to plan and implement parts of a small backbone network based on the specification of requirements.,JAMK,TT00CE28,Core Networks
2,The student knows how to solve local area network problems.,JAMK,TT00CE20,Local Area Networks
3,The student knows how to use communication and reporting tools in accordance with the instructions.,JAMK,TT00CE12,Security Operations Center
4,You can apply your knowledge to the solutions of practical tasks.,JAMK,TT00CD77,Basics of Programming
5,You can apply your knowledge to the solutions of practical tasks.,JAMK,TT00CD91,Object Oriented Programming
6,You have completed all the assignments of the course.,JAMK,TT00CD76,Scripting and Automatization
7,You have completed all the assignments of the course.,JAMK,TT00CD77,Basics of Programming
8,You have completed all the assignments of the course.,JAMK,TT00CD94,Mobile Application Development
9,You have completed almost all the assignments of the course.,JAMK,TT00CD78,Frontend Programming


### Interpretation of repeated assessment fragments

The repeated-fragment audit shows that `assessment_text` contains a mixture of:

- **course-specific semantic content**, such as domain methods, technologies, and discipline-specific expectations
- **repeated rubric language**, such as assignment completion, general performance level, independent work, and reporting behavior

This does not make the full variant invalid. However, it does mean that `embedding_text_full` should be treated as a **controlled experimental variant**, not as an automatically superior representation.

In practice, the full variant may help when assessment criteria contain meaningful domain-specific scope information, but it may also introduce additional similarity caused by shared pedagogical boilerplate across otherwise different courses. For that reason, later benchmark notebooks determine empirically whether the added assessment text improves overlap detection or mainly adds noise.

Manual filtering of repeated rubric language is not applied in this notebook. Such filtering would require subjective decisions about which phrases count as boilerplate and would introduce handcrafted weighting that is inconsistent with the conservative, label-free preprocessing design used here.

### 10. Sample inspection of the final course representation

A preprocessing notebook should include human-readable examples.

This section prints selected courses so that the generated representation can be reviewed directly instead of treated as a black box.

In [72]:
# Cell 28 - Inspect representative course records across different representation patterns

representative_codes = [
    "TT00CD70",  # Data Networks - common fields + prerequisites + assessment
    "TT00CD71",  # Linux Basics - common fields without prerequisites
    "TT00CD96",  # Graphics Programming - learning outcomes
    "TT00CD90",  # Software Design - multidisciplinary + communication
    "TT00CD88",  # Hardening - applying technology to practice
    "TT00CE09",  # Auditing and Penetration Testing - investigations + multidisciplinary
]

inspection_df = df[df["course_unit_code"].isin(representative_codes)].copy()

for _, row in inspection_df.iterrows():
    print("=" * 120)
    print(f"Course: {row['institution']} | {row['course_unit_code']} | {row['course_name']}")
    print("-" * 120)
    print("embedding_text_main")
    print(row["embedding_text_main"])
    print()
    print("embedding_text_enriched")
    print(row["embedding_text_enriched"])
    print()
    print("embedding_text_full")
    print(row["embedding_text_full"])
    print()

Course: JAMK | TT00CD70 | Data Networks
------------------------------------------------------------------------------------------------------------------------
embedding_text_main
Data Networks
You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
You 

In [73]:
# Cell 28B - Compare review text for edge-case courses

edge_case_codes = [
    "TT00CD96",  # Graphics Programming
    "TT00CD90",  # Software Design
    "TT00CD88",  # Hardening
    "TT00CE09",  # Auditing and Penetration Testing
    "TT00CD94",  # JAMK Mobile Application Development
    "TT00DS75",  # Turku AMK Mobile Application Development
]

edge_case_df = df[df["course_unit_code"].isin(edge_case_codes)].copy()

for _, row in edge_case_df.iterrows():
    print("=" * 120)
    print(f"Course: {row['institution']} | {row['course_unit_code']} | {row['course_name']}")
    print("-" * 120)
    print(row["review_text_full"])
    print()

Course: JAMK | TT00CD90 | Software Design
------------------------------------------------------------------------------------------------------------------------
Institution: JAMK
Course code: TT00CD90
Course name: Software Design
Objective: After completing the course, you will learn the principles, methods and tools of software engineering. The primary goal is to strengthen the definition and service design skills and to teach how to document the customer's needs in such a way that they are useful for the product development team. You will learn to use the commonly used UML (Unified Modeling Language) description language. The UML description method is introduced to the most common uses, such as use case (Use Case), sequence diagram (Sequence Diagram), state machine description (State Chart) or deployment diagram (Deployment Diagram). To help with the definition, the use of so-called MockUp tool is learned.
Knowledge and understanding: The student knows how to define software soluti

### 11. Assemble the final cleaned dataframe

The saved output should support:
- later notebooks
- manual labeling
- benchmark experiments
- the final proof-of-concept UI

For that reason, the saved dataset includes:
- cleaned descriptive fields
- normalized helper fields
- embedding text variants
- review text fields
- reference metadata

In [74]:
# Cell 29 - Select final columns for the cleaned dataset

final_columns = [
    "institution",
    "degree_programme",
    "course_name",
    "course_unit_code",
    "credits",
    "teaching_language_text",
    "responsible_person_text",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_grading_scale",
    "assessment_text",
    "course_name_normalized",
    "prerequisites_text_normalized",
    "embedding_text_main",
    "embedding_text_enriched",
    "embedding_text_full",
    "review_text_compact",
    "review_text_full",
]

clean_df = df[final_columns].copy()

print("Clean dataframe shape:", clean_df.shape)
clean_df.head(3)

Clean dataframe shape: (50, 26)


,institution,degree_programme,course_name,course_unit_code,credits,teaching_language_text,responsible_person_text,objective_text,competences_knowledge_text,competences_practice_text,competences_applying_technology_text,competences_multidisciplinary_text,competences_communication_text,competences_investigations_text,learning_outcomes_text,content_text,prerequisites_text,assessment_grading_scale,assessment_text,course_name_normalized,prerequisites_text_normalized,embedding_text_main,embedding_text_enriched,embedding_text_full,review_text_compact,review_text_full
0,JAMK,Information and Communication Technology,Data Networks,TT00CD70,5,"Finnish, English",Pasi Hyytiäinen,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...,You can apply data network skills to communication between devices.,,,,,,"In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication. You will master...",Linux Basics,1-5,Grade 1: The student knows the theory on network protocols sufficiently and is able to recognize some sections of existing solutions. The student can design...,data networks,linux basics,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Institution: JAMK\nCourse code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers an...,Institution: JAMK\nCourse code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers an...
1,JAMK,Information and Communication Technology,Linux Basics,TT00CD71,4,"Finnish, English","Juho Pekki, Teemu Siikaniemi","After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage t...",You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.,You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions....,,,,,,"In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be ab...",,1-5,"Grade 1: You are familiar with the basic concepts of the Linux operating system and can use basic commands. You know how to create user groups, add users to...",linux basics,,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Institution: JAMK\nCourse code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts...","Institution: JAMK\nCourse code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts..."
2,JAMK,Informati

### 12. Final diagnostics before saving

These checks confirm that the cleaned output is complete enough for downstream use.

The focus is on:
- empty critical fields
- duplicate identifiers
- representative text lengths
- overall readiness for Notebook 02 and Notebook 03

In [75]:
# Cell 30 - Check critical emptiness after final assembly

critical_empty_summary = {
    "empty_institution": (clean_df["institution"].fillna("").str.strip() == "").sum(),
    "empty_course_name": (clean_df["course_name"].fillna("").str.strip() == "").sum(),
    "empty_course_unit_code": (clean_df["course_unit_code"].fillna("").str.strip() == "").sum(),
    "empty_objective_text": (clean_df["objective_text"].fillna("").str.strip() == "").sum(),
    "empty_content_text": (clean_df["content_text"].fillna("").str.strip() == "").sum(),
    "empty_embedding_text_main": (clean_df["embedding_text_main"].fillna("").str.strip() == "").sum(),
    "empty_embedding_text_enriched": (clean_df["embedding_text_enriched"].fillna("").str.strip() == "").sum(),
    "empty_embedding_text_full": (clean_df["embedding_text_full"].fillna("").str.strip() == "").sum(),
    "empty_review_text_full": (clean_df["review_text_full"].fillna("").str.strip() == "").sum(),
}

pd.Series(critical_empty_summary)

empty_institution                0
empty_course_name                0
empty_course_unit_code           0
empty_objective_text             0
empty_content_text               0
empty_embedding_text_main        0
empty_embedding_text_enriched    0
empty_embedding_text_full        0
empty_review_text_full           0
dtype: int64

In [76]:
# Cell 31 - Final text length summary for the main representations

final_length_summary = clean_df[[
    "embedding_text_main",
    "embedding_text_enriched",
    "embedding_text_full",
    "review_text_full",
]].copy()

for col in final_length_summary.columns:
    final_length_summary[f"{col}_char_len"] = final_length_summary[col].apply(char_len)
    final_length_summary[f"{col}_word_len"] = final_length_summary[col].apply(word_len)

final_length_summary[
    [c for c in final_length_summary.columns if c.endswith("_char_len") or c.endswith("_word_len")]
].describe()

,embedding_text_main_char_len,embedding_text_main_word_len,embedding_text_enriched_char_len,embedding_text_enriched_word_len,embedding_text_full_char_len,embedding_text_full_word_len,review_text_full_char_len,review_text_full_word_len
count,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,1506.020000,218.08000,1616.660000,232.500000,2811.140000,419.640000,2925.840000,433.140000
std,550.003951,75.47478,612.904347,85.708393,1072.285657,164.146868,1080.667261,165.014534
min,368.000000,50.00000,368.000000,50.000000,368.000000,50.000000,446.000000,60.000000
25%,1209.500000,178.25000,1274.750000,190.750000,2420.250000,367.750000,2555.250000,380.500000
50%,1419.000000,209.00000,1507.500000,219.500000,2702.500000,413.000000,2831.500000,428.000000
75%,1727.000000,245.00000,1921.000000,266.750000,3460.750000,503.000000,3588.500000,518.000000
max,3382.000000,420.00000,3382.000000,430.000000,4869.000000,731.000000,4989.000000,745.000000


In [77]:
# Cell 31B - Token count estimate vs model context limits

token_estimate_rows = []

for variant in ["embedding_text_main", "embedding_text_enriched", "embedding_text_full"]:
    word_counts = clean_df[variant].apply(word_len)

    # Conservative rough estimate for English-heavy educational text
    estimated_tokens = word_counts * 1.3

    token_estimate_rows.append({
        "variant": variant,
        "mean_words": round(word_counts.mean(), 1),
        "max_words": int(word_counts.max()),
        "mean_tokens_est": round(estimated_tokens.mean(), 0),
        "max_tokens_est": round(estimated_tokens.max(), 0),
        "pct_courses_over_512": round((estimated_tokens > 512).mean() * 100, 1),
        "pct_courses_over_8192": round((estimated_tokens > 8192).mean() * 100, 1),
    })

token_context_df = pd.DataFrame(token_estimate_rows)

context_limits_df = pd.DataFrame([
    {"model": "multilingual-e5-base", "context_tokens": 512},
    {"model": "multilingual-e5-large", "context_tokens": 512},
    {"model": "multilingual-e5-large-instruct", "context_tokens": 512},
    {"model": "BAAI/bge-m3", "context_tokens": 8192},
    {"model": "OpenAI text-embedding-3-large", "context_tokens": 8192},
    {"model": "TF-IDF", "context_tokens": "N/A"},
    {"model": "BM25", "context_tokens": "N/A"},
])

print("Token count estimates by variant:")
display(token_context_df)

print("\nModel context limits used in later benchmarking:")
display(context_limits_df)

Token count estimates by variant:


,variant,mean_words,max_words,mean_tokens_est,max_tokens_est,pct_courses_over_512,pct_courses_over_8192
0,embedding_text_main,218.1,420,284.0,546.0,2.0,0.0
1,embedding_text_enriched,232.5,430,302.0,559.0,8.0,0.0
2,embedding_text_full,419.6,731,546.0,950.0,58.0,0.0



Model context limits used in later benchmarking:


,model,context_tokens
0,multilingual-e5-base,512
1,multilingual-e5-large,512
2,multilingual-e5-large-instruct,512
3,BAAI/bge-m3,8192
4,OpenAI text-embedding-3-large,8192
5,TF-IDF,N/A
6,BM25,N/A


#### Interpretation of context-window constraints for later benchmarking

The token-count audit shows that the three text variants place clearly different demands on embedding models.

- `embedding_text_main` has an estimated mean length of about 283 tokens and exceeds 512 tokens for only 2% of courses.
- `embedding_text_enriched` has an estimated mean length of about 301 tokens and exceeds 512 tokens for 8% of courses.
- `embedding_text_full` has an estimated mean length of about 541 tokens and exceeds 512 tokens for 58% of courses.

This has direct implications for later benchmark interpretation. Results from 512-token embedding models should be interpreted cautiously especially on the full variant, because truncation may remove part of the course representation. By contrast, long-context models such as `BAAI/bge-m3` and `text-embedding-3-large` can process all three variants without practical context-window concerns in this dataset.

For that reason, later notebooks should treat:
- `embedding_text_main` as the fairest short-context benchmark representation
- `embedding_text_enriched` as a moderate extension with limited truncation risk
- `embedding_text_full` as a controlled long-text experimental condition

In [78]:
# Cell 32 - Save cleaned dataset to CSV

clean_df.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8")

print("Saved cleaned dataset to:", CLEAN_CSV_PATH.resolve())
print("Rows, columns:", clean_df.shape)

Saved cleaned dataset to: C:\Users\user\Downloads\curriculum-overlap-poc\data\courses_clean.csv
Rows, columns: (50, 26)


In [79]:
# Cell 33 - Final preview of the most important output columns

clean_df[[
    "institution",
    "course_unit_code",
    "course_name",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_text",
    "embedding_text_main",
    "embedding_text_enriched",
    "embedding_text_full",
    "review_text_full",
]].head(5)

,institution,course_unit_code,course_name,objective_text,competences_knowledge_text,competences_practice_text,competences_applying_technology_text,competences_multidisciplinary_text,competences_communication_text,competences_investigations_text,learning_outcomes_text,content_text,prerequisites_text,assessment_text,embedding_text_main,embedding_text_enriched,embedding_text_full,review_text_full
0,JAMK,TT00CD70,Data Networks,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...,You can apply data network skills to communication between devices.,,,,,,"In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication. You will master...",Linux Basics,Grade 1: The student knows the theory on network protocols sufficiently and is able to recognize some sections of existing solutions. The student can design...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Institution: JAMK\nCourse code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers an...
1,JAMK,TT00CD71,Linux Basics,"After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage t...",You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.,You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions....,,,,,,"In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be ab...",,"Grade 1: You are familiar with the basic concepts of the Linux operating system and can use basic commands. You know how to create user groups, add users to...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Institution: JAMK\nCourse code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts..."
2,JAMK,TT00CD72,Servers and containers,You understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud computing. You are able ...,You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service models. You understand the core pri...,"You are able to utilize basic cloud services and create virtual machines. You are able to build, run, and manage containers, and use orchestration tools to ...",,,,,,"In this course, you will learn the basic concepts of servers and cloud computing, and differentiate between different cloud service models. You will master ...","Linux Basics, Data Networks, Databases"

## Final note

The dataset combines courses from more than one institution in order to create a more informative overlap benchmark. This choice increases variation in course descriptions and reduces the risk that the evaluation corpus would contain too few meaningful overlap cases if it were restricted to a narrower single-institution set. Institution is therefore preserved as reference metadata for inspection and labeling, but it is intentionally excluded from embedding text so that similarity reflects curricular meaning rather than source identity.

This notebook performs conservative and thesis-defensible preprocessing for curriculum-overlap detection.

### What this notebook does
- preserves the real course dataset structure
- flattens nested JSON into reusable tabular fields
- normalizes whitespace and simple formatting noise
- creates three embedding-ready text variants
- prepares fair review text for later human labeling
- audits field coverage, text length, enrichment impact, assessment impact, and repetitiveness
- saves a cleaned dataset for downstream notebooks and the PoC UI

### What this notebook intentionally does not do
- remove stop words
- aggressively rewrite academic language
- inject prerequisite course descriptions into the main text
- add administrative metadata such as institution, degree programme, or course code into embedding text
- apply handcrafted weighting tricks at preprocessing time

### Why this design is appropriate

The objective of the thesis is not to force a specific model to win.  
The objective is to represent each course as faithfully as possible for the task of curriculum overlap detection.

The selected representation therefore emphasizes:
- course objectives
- common competence statements
- course content
- additional curriculum signals only when they exist and are potentially meaningful
- assessment criteria only as a controlled extended condition

The main representation, `embedding_text_main`, is intentionally conservative.  
It provides a stable and comparable benchmark representation across the full dataset.

The enriched representation, `embedding_text_enriched`, preserves sparse but semantically meaningful curriculum signals such as:
- learning outcomes
- teamwork-related competence statements
- multidisciplinary competence statements
- investigations/information retrieval statements
- prerequisite relations

The full representation, `embedding_text_full`, extends this design further by adding assessment criteria text through the combined `assessment_text` field.  
This allows later notebooks to test whether course-specific rubric language improves overlap detection, while also making it possible to observe whether such text introduces redundancy or boilerplate effects.

These broader signals are not assumed to improve results automatically.  
Instead, they are retained as controlled comparisons so that later notebooks can test whether broader curriculum context improves overlap detection.

In particular, the embedding variants are kept largely **label-free** in order to reduce **structural bias** caused by repeated formatting tokens shared across all course descriptions. At the same time, light semantic cues are retained where relationship meaning matters, especially for prerequisites, learning outcomes, and assessment criteria.

Institution is preserved as reference metadata and review context, but intentionally excluded from embedding text so that the benchmark focuses on curricular meaning rather than source identity.

This keeps the pipeline transparent, reproducible, and suitable for both academic evaluation and practical proof-of-concept use.